In [1]:
                           ####  Rebuild and push preprocessing image to ECR ####

In [2]:
# Account
!aws sts get-caller-identity

{
    "UserId": "AROA6MGH2ZGRWOHARXCZY:SageMaker",
    "Account": "988261566883",
    "Arn": "arn:aws:sts::988261566883:assumed-role/SageMakerStudioExecutionRole2026/SageMaker"
}


In [3]:
#Definir variables
AWS_ACCOUNT_ID = "988261566883"
AWS_REGION = "us-east-1"
ECR_REPO_NAME = "ml-preprocessing"
IMAGE_TAG = "pipeline-v1"

ECR_URI = f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/{ECR_REPO_NAME}:{IMAGE_TAG}"
ECR_URI

'988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v1'

In [4]:
# Container creation
!aws ecr create-repository --repository-name {ECR_REPO_NAME} --region {AWS_REGION}


aws: [ERROR]: An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'ml-preprocessing' already exists in the registry with id '988261566883'

Additional error details:
message: The repository with name 'ml-preprocessing' already exists in the registry with id '988261566883'


In [5]:
# Build image
!docker build --network sagemaker -t ml-preprocessing:{IMAGE_TAG} -f src/preprocessing/Dockerfile .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  244.5MB
Step 1/8 : FROM python:3.13-slim
3.13-slim: Pulling from library/python

1dee3f47: Pulling fs layer 
029d0aef: Pulling fs layer 
a5e6ffbf: Pulling fs layer 
Digest: sha256:739e7213785e88c0f702dcdc12c0973afcbd606dbf021a589cab77d6b00b579d
Status: Downloaded newer image for python:3.13-slim
 ---> 9aafbb8a9ec1
Step 2/8 : WORKDIR /app
 ---> Running in e0975366c7cb
 ---> Removed intermediate container e0975366c7cb
 ---> 7d0114cf71dd
Step 3/8 : COPY src/preprocessing/requirements.txt ./requirements.txt
 ---> 038812e02d75
Step 4/8 : RUN pip install --no-cache-dir -r requirements.txt
 ---> Running in 6bc06828fbd7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 361.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 

In [6]:
# Loggin ECR
!aws ecr get-login-password --region {AWS_REGION} | docker login --username AWS --password-stdin {AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded


In [7]:
# Tag Local Image
!docker tag ml-preprocessing:{IMAGE_TAG} {ECR_URI}

In [8]:
# Image Push
!docker push {ECR_URI} --quiet

988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v1


In [9]:
# Validation
!aws ecr describe-images --repository-name {ECR_REPO_NAME} --region {AWS_REGION} --image-ids imageTag={IMAGE_TAG}

{
    "imageDetails": [
        {
            "registryId": "988261566883",
            "repositoryName": "ml-preprocessing",
            "imageDigest": "sha256:30733dd4feaecfe4bfa67d6a4e9b67c979cf0a44db01dfc42e7737b52aaecaf2",
            "imageTags": [
                "pipeline-v1"
            ],
            "imageSizeInBytes": 575184920,
            "imagePushedAt": "2026-03-31T18:33:15.024000+00:00",
            "imageManifestMediaType": "application/vnd.docker.distribution.manifest.v2+json",
            "artifactMediaType": "application/vnd.docker.container.image.v1+json",
            "imageStatus": "ACTIVE"
        }
    ]
}


In [10]:
# SAVING THE PREPROCESING URI
preprocess_image_uri = f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/{ECR_REPO_NAME}:{IMAGE_TAG}"
preprocess_image_uri

'988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v1'

In [11]:
                            ####  Rebuild and push trainning image to ECR ####

In [12]:
# Define ECR variables for the training image
TRAIN_ECR_REPO_NAME = "ml-training"
TRAIN_IMAGE_TAG = "pipeline-v1"

train_image_uri = (
    f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/"
    f"{TRAIN_ECR_REPO_NAME}:{TRAIN_IMAGE_TAG}"
)

train_image_uri

'988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-training:pipeline-v1'

In [13]:
# Build the training container image
!docker build --network sagemaker -t {TRAIN_ECR_REPO_NAME}:{TRAIN_IMAGE_TAG} -f src/training/Dockerfile .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  244.5MB
Step 1/11 : FROM python:3.12-slim
3.12-slim: Pulling from library/python

1dee3f47: Already exists 
0e2fae4c: Pulling fs layer 
911b36ac: Pulling fs layer 
Digest: sha256:3d5ed973e45820f5ba5e46bd065bd88b3a504ff0724d85980dcd05eab361fcf4
Status: Downloaded newer image for python:3.12-slim
 ---> fb1118f126b5
Step 2/11 : ENV PYTHONDONTWRITEBYTECODE=1
 ---> Running in 7df91847da64
 ---> Removed intermediate container 7df91847da64
 ---> 63d79a800c3f
Step 3/11 : ENV PYTHONUNBUFFERED=1
 ---> Running in 9cfc39697bc3
 ---> Removed intermediate container 9cfc39697bc3
 ---> 993af3bf6019
Step 4/11 : WORKDIR /opt/ml/code
 ---> Running in 1fe7629fac85
 ---> Removed intermediate container 1fe7629fac85
 ---> 436c0780e10f
Step 5/11 : COPY src/training

In [14]:
# Check if the ECR repository exists
!aws ecr describe-repositories --region {AWS_REGION}

{
    "repositories": [
        {
            "repositoryArn": "arn:aws:ecr:us-east-1:988261566883:repository/sagemaker-dask-example",
            "registryId": "988261566883",
            "repositoryName": "sagemaker-dask-example",
            "repositoryUri": "988261566883.dkr.ecr.us-east-1.amazonaws.com/sagemaker-dask-example",
            "createdAt": "2026-03-07T16:43:47.857000+00:00",
            "imageTagMutability": "MUTABLE",
            "imageScanningConfiguration": {
                "scanOnPush": false
            },
            "encryptionConfiguration": {
                "encryptionType": "AES256"
            }
        },
        {
            "repositoryArn": "arn:aws:ecr:us-east-1:988261566883:repository/demand-forecasting-training",
            "registryId": "988261566883",
            "repositoryName": "demand-forecasting-training",
            "repositoryUri": "988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-training",
            "createdAt": "2026-03

In [15]:
# Create repository if it does not exist
!aws ecr create-repository --repository-name {TRAIN_ECR_REPO_NAME} --region {AWS_REGION}


aws: [ERROR]: An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'ml-training' already exists in the registry with id '988261566883'

Additional error details:
message: The repository with name 'ml-training' already exists in the registry with id '988261566883'


In [16]:
# Authenticate Docker with ECR
!aws ecr get-login-password --region {AWS_REGION} | docker login --username AWS --password-stdin {AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded


In [17]:
# Tag the local image with ECR URI
!docker tag {TRAIN_ECR_REPO_NAME}:{TRAIN_IMAGE_TAG} {train_image_uri}

In [18]:
# Push the image to ECR
!docker push {train_image_uri} --quiet

988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-training:pipeline-v1


In [19]:
# Verify the image was successfully pushed
!aws ecr describe-images \
  --repository-name {TRAIN_ECR_REPO_NAME} \
  --region {AWS_REGION} \
  --image-ids imageTag={TRAIN_IMAGE_TAG}

{
    "imageDetails": [
        {
            "registryId": "988261566883",
            "repositoryName": "ml-training",
            "imageDigest": "sha256:0c16aeaaf09527fac725baaf5566fb54daa3aa6658f42388d6cf1a1b821158c0",
            "imageTags": [
                "pipeline-v1"
            ],
            "imageSizeInBytes": 575984650,
            "imagePushedAt": "2026-03-31T18:35:12.948000+00:00",
            "imageManifestMediaType": "application/vnd.docker.distribution.manifest.v2+json",
            "artifactMediaType": "application/vnd.docker.container.image.v1+json",
            "imageStatus": "ACTIVE"
        }
    ]
}


In [20]:
# Save train URI
train_image_uri = (
    f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/"
    f"{TRAIN_ECR_REPO_NAME}:{TRAIN_IMAGE_TAG}"
)
train_image_uri

'988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-training:pipeline-v1'

In [21]:
                            ####  Build and push evaluate image to ECR ####

In [22]:
# Define ECR variables for the evaluation image
EVAL_ECR_REPO_NAME = "ml-preprocessing"
EVAL_IMAGE_TAG = "pipeline-v2"

eval_image_uri = (
    f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/"
    f"{EVAL_ECR_REPO_NAME}:{EVAL_IMAGE_TAG}"
)

eval_image_uri

'988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v2'

In [23]:
# Build the evaluation container image
!docker build --network sagemaker -t {EVAL_ECR_REPO_NAME}:{EVAL_IMAGE_TAG} -f src/preprocessing/Dockerfile .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  244.5MB
Step 1/8 : FROM python:3.13-slim
 ---> 9aafbb8a9ec1
Step 2/8 : WORKDIR /app
 ---> Using cache
 ---> 7d0114cf71dd
Step 3/8 : COPY src/preprocessing/requirements.txt ./requirements.txt
 ---> Using cache
 ---> 038812e02d75
Step 4/8 : RUN pip install --no-cache-dir -r requirements.txt
 ---> Using cache
 ---> b96b8ea8ece9
Step 5/8 : COPY src ./src
 ---> Using cache
 ---> dcdeb249ad0c
Step 6/8 : ENTRYPOINT ["python", "-m", "src.preprocessing"]
 ---> Using cache
 ---> d7e51571aa66
Step 7/8 : CMD ["--raw-path", "data/raw", "--output-path", "data/prep"]
 ---> Using cache
 ---> 916a2b65db12
Step 8/8 : LABEL com.amazon.studio.user.resources=true
 ---> Using cache
 ---> 71c07629724d
Successfully built 71c07629724d
Successfully tagged ml-preproce

In [24]:
# Authenticate Docker with ECR
!aws ecr get-login-password --region {AWS_REGION} | docker login --username AWS --password-stdin {AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded


In [25]:
# Tag the local image with ECR URI
!docker tag {EVAL_ECR_REPO_NAME}:{EVAL_IMAGE_TAG} {eval_image_uri}

In [26]:
# Push the image to ECR
!docker push {eval_image_uri} --quiet

988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v2


In [27]:
# Verify the image was successfully pushed
!aws ecr describe-images \
  --repository-name {EVAL_ECR_REPO_NAME} \
  --region {AWS_REGION} \
  --image-ids imageTag={EVAL_IMAGE_TAG}

{
    "imageDetails": [
        {
            "registryId": "988261566883",
            "repositoryName": "ml-preprocessing",
            "imageDigest": "sha256:30733dd4feaecfe4bfa67d6a4e9b67c979cf0a44db01dfc42e7737b52aaecaf2",
            "imageTags": [
                "pipeline-v1",
                "pipeline-v2"
            ],
            "imageSizeInBytes": 575184920,
            "imagePushedAt": "2026-03-31T18:33:15.024000+00:00",
            "imageManifestMediaType": "application/vnd.docker.distribution.manifest.v2+json",
            "artifactMediaType": "application/vnd.docker.container.image.v1+json",
            "imageStatus": "ACTIVE"
        }
    ]
}


In [28]:
# VERIFICATION OF IMAGES
print(preprocess_image_uri)
print(train_image_uri)
print(eval_image_uri)

988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v1
988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-training:pipeline-v1
988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v2


In [29]:
                         ####  Build and push inference image to ECR ####

In [30]:
# Define ECR variables for inference image
SERVING_ECR_REPO_NAME = "ml-inference"
SERVING_IMAGE_TAG = "pipeline-v1"

serving_image_uri = (
    f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/"
    f"{SERVING_ECR_REPO_NAME}:{SERVING_IMAGE_TAG}"
)

serving_image_uri

'988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-inference:pipeline-v1'

In [31]:
# Build inference container image
!docker build --network sagemaker -t {SERVING_ECR_REPO_NAME}:{SERVING_IMAGE_TAG} -f src/inference/Dockerfile .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  244.5MB
Step 1/11 : FROM python:3.12-slim
 ---> fb1118f126b5
Step 2/11 : ENV PYTHONDONTWRITEBYTECODE=1
 ---> Using cache
 ---> 63d79a800c3f
Step 3/11 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> 993af3bf6019
Step 4/11 : WORKDIR /opt/ml/code
 ---> Using cache
 ---> 436c0780e10f
Step 5/11 : COPY src/inference/requirements.txt /opt/ml/code/requirements.txt
 ---> 23395f2bc51c
Step 6/11 : RUN pip install --no-cache-dir --upgrade pip &&     pip install --no-cache-dir -r requirements.txt
 ---> Running in 848738f45104
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1
   ━━━━━━━━━

In [32]:
# Create repository if needed
!aws ecr create-repository \
  --repository-name {SERVING_ECR_REPO_NAME} \
  --region {AWS_REGION}


aws: [ERROR]: An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'ml-inference' already exists in the registry with id '988261566883'

Additional error details:
message: The repository with name 'ml-inference' already exists in the registry with id '988261566883'


In [33]:
# Login to ECR
!aws ecr get-login-password --region {AWS_REGION} | docker login --username AWS --password-stdin {AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded


In [34]:
# Tag image
!docker tag {SERVING_ECR_REPO_NAME}:{SERVING_IMAGE_TAG} {serving_image_uri}

In [35]:
# Push image
!docker push {serving_image_uri} --quiet

988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-inference:pipeline-v1


In [36]:
# Validate image
!aws ecr describe-images \
  --repository-name {SERVING_ECR_REPO_NAME} \
  --region {AWS_REGION} \
  --image-ids imageTag={SERVING_IMAGE_TAG}

{
    "imageDetails": [
        {
            "registryId": "988261566883",
            "repositoryName": "ml-inference",
            "imageDigest": "sha256:0f55e459f6b3f8f551dc733ba835ddf608e55a1c3745bb253e67095f8b66d650",
            "imageTags": [
                "pipeline-v1"
            ],
            "imageSizeInBytes": 577525336,
            "imagePushedAt": "2026-03-31T18:37:10.421000+00:00",
            "imageManifestMediaType": "application/vnd.docker.distribution.manifest.v2+json",
            "artifactMediaType": "application/vnd.docker.container.image.v1+json",
            "imageStatus": "ACTIVE"
        }
    ]
}


In [37]:
# VERIFICATION OF IMAGES
print(preprocess_image_uri)
print(train_image_uri)
print(eval_image_uri)
print (serving_image_uri)

988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v1
988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-training:pipeline-v1
988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v2
988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-inference:pipeline-v1


In [38]:
                                          " ----------------- "

' ----------------- '

In [39]:
# Correr al regresar
#AWS_ACCOUNT_ID = "988261566883"
#AWS_REGION = "us-east-1"

#preprocess_image_uri = "988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v1"
#train_image_uri = "988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-training:pipeline-v1"
#eval_image_uri = "988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing:pipeline-v2"
#serving_image_uri = "988261566883.dkr.ecr.us-east-1.amazonaws.com/ml-inference:pipeline-v1"